# Cascade: the evaluator as a PyTorch model

We already have eight extracted features and a weighted evaluator. This notebook puts that evaluator inside PyTorch, starting from the existing weights.

Start JupyterLab from `part_b` using the command in `training/README.md`, then open this notebook with **Python (Cascade)**. The notebook runs from `training/td_lambda`, so the project root is `../..`. Run cells from top to bottom with **Shift+Enter**.

This first step covers tensors, `nn.Linear`, a forward pass and automatic differentiation. We will choose delayed-reward training targets in the next step. Predictions here are heuristic scores, not win probabilities.

In [ ]:
from pathlib import Path

import numpy as np
import torch
from numpy.typing import NDArray
from torch import nn

from training.baseline import BASELINE_PROFILE_NAMES, BASELINE_WEIGHTS, FEATURE_NAMES
from training.DataExtraction.extract_features import FeatureGame, load_features

project_root: Path = Path("../..")


## Load one recorded game

`feature_path` is a `Path` to an extracted **feature archive**. `game` is the loaded `FeatureGame` dictionary. Its `features` array contains a vector for each board and each player perspective.

The chosen file is the recorded short-budget game against teacher level 3. Change this path to inspect another feature archive. We retain the game outcome as metadata and do not convert it into a target for every position.

In [ ]:
feature_path: Path = project_root / "logs/shortBudget-TeacherL3L4/features/game-000001.npz"
game: FeatureGame = load_features(feature_path)

assert tuple(game["feature_names"].tolist()) == tuple(FEATURE_NAMES)
assert game["perspectives"].tolist() == ["RED", "BLUE"]
print("Game:", game["game_id"])
print("Outcome:", game["winner"], game["ending_reason"])
print("Feature array:", game["features"].shape, game["features"].dtype)
print("Columns:", FEATURE_NAMES)

## Make a tensor batch

Keep nonterminal PLAY positions in chronological order and use **Red's perspective** throughout. Red's perspective is independent of whose turn it is.

`features_np` is a NumPy `float32` array with shape $(M, 8)$. `X` is a PyTorch `float32` tensor with the same shape. `position_indices` maps its rows back to the original game history.

[`torch.from_numpy`](https://docs.pytorch.org/docs/stable/generated/torch.from_numpy.html) shares storage with the array it receives. We copy the selected features first, so changes to `X` cannot alter `game["features"]`.

In [ ]:
play_mask: NDArray[np.bool_] = (game["phase"] == "PLAY") & np.logical_not(game["is_terminal"])
position_indices: NDArray[np.int32] = game["position_index"][play_mask].copy()
features_np: NDArray[np.float32] = game["features"][play_mask, 0, :].copy()
X: torch.Tensor = torch.from_numpy(features_np)

assert X.dtype == torch.float32
assert X.ndim == 2 and X.shape[1] == len(FEATURE_NAMES)
assert X.shape[0] > 0
print("X:", X.shape, X.dtype)
print("First row is board index:", int(position_indices[0]))
X[:3]

## Use PyTorch's built-in linear layer

Our evaluator is

$$
V_{\mathbf w}(s)=\sum_{i=1}^{8} w_i f_i(s).
$$

[`nn.Linear`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html) represents this directly. Eight inputs produce one scalar per position. `bias=False` gives us the same weighted-sum structure as the current evaluator.

PyTorch stores this layer's weights with shape $(1, 8)$. Initialise them from the frozen `phalanx_aggressive` profile. [`torch.no_grad`](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html) keeps this initialisation outside the gradient calculation. All eight coefficients are trainable, including alignment even though it starts at zero.

In [ ]:
profile_name: str = "phalanx_aggressive"
profile_index: int = BASELINE_PROFILE_NAMES.index(profile_name)
initial_weights_np: NDArray[np.float32] = BASELINE_WEIGHTS[profile_index].copy()
initial_weights: torch.Tensor = torch.from_numpy(initial_weights_np)

model: nn.Linear = nn.Linear(in_features=len(FEATURE_NAMES), out_features=1, bias=False)
with torch.no_grad():
    model.weight.copy_(initial_weights.unsqueeze(0))

print(model)
print("Weight shape:", model.weight.shape)
print("Trainable:", model.weight.requires_grad)
list(zip(FEATURE_NAMES, model.weight.detach()[0].tolist()))

## Run a forward pass

Calling `model(X)` evaluates every row in the batch. Its output has shape $(M, 1)$: one score per position.

Check the result against the weighted sum of the same stored features in NumPy. Small rounding differences are expected with `float32`. This checks that we represented the evaluator correctly; it does not assess playing strength.

In [ ]:
predictions: torch.Tensor = model(X)
reference_np: NDArray[np.float32] = (features_np * initial_weights_np).sum(axis=1, keepdims=True)
reference: torch.Tensor = torch.from_numpy(reference_np)

torch.testing.assert_close(predictions.detach(), reference, rtol=1e-5, atol=1e-5)
print("Predictions:", predictions.shape)
print("Gradient tracking:", predictions.requires_grad)
print("Maximum difference:", float((predictions.detach() - reference).abs().max()))
predictions[:5]

## Inspect automatic differentiation

PyTorch tracks how the score depends on `model.weight`. For a linear evaluator,

$$
\frac{\partial V_{\mathbf w}(s)}{\partial w_i}=f_i(s).
$$

Ask PyTorch to differentiate the first position's score and inspect `model.weight.grad`. It should equal that position's feature vector. `backward()` computes gradients; it does not change the weights.

We differentiate a score here to inspect autograd. Actual training will differentiate a loss against a target.

In [ ]:
model.zero_grad(set_to_none=True)
first_score: torch.Tensor = model(X[:1]).squeeze()
first_score.backward()

assert model.weight.grad is not None
torch.testing.assert_close(model.weight.grad, X[:1])
torch.testing.assert_close(model.weight.detach()[0], initial_weights)
print("Gradient shape:", model.weight.grad.shape)
list(zip(FEATURE_NAMES, model.weight.grad[0].tolist()))

## Prepare the built-in training components

These are the actual PyTorch objects we will use next: a mean squared error loss and the Adam optimiser. Adam receives the layer's trainable parameters. We have not called `optimizer.step()` yet.

The next step is to construct targets from game sequences and delayed rewards. We need that decision before there is a meaningful loss to optimise.

In [ ]:
loss_fn: nn.MSELoss = nn.MSELoss()
optimizer: torch.optim.Adam = torch.optim.Adam(model.parameters(), lr=1e-3)
optimizer.zero_grad(set_to_none=True)

print("Loss:", loss_fn)
print("Optimiser:", type(optimizer).__name__)
print("Learning rate:", optimizer.param_groups[0]["lr"])

## Your next cell

Inspect `X[0] * model.weight.detach()[0]` to see each feature's contribution to the first score. Pair the values with `FEATURE_NAMES`.

Which feature contributes most to that score? How is a feature's contribution different from the gradient we inspected above?

In [ ]:
# Inspect the individual feature contributions here.